# Part 1. Equation of a Slime

How many late days are you using for this assignment?
0 days

In [ ]:
# Imports section
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

## 1. Loading the dataset

In [1]:
# Using pandas load the dataset
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)
science = pd.read_csv("science_data_large.csv")


print("Science_data_large CSV File:")
print(science.head(15).to_string(),"\n")


NameError: name 'pd' is not defined

## 2. Splitting the dataset

In [ ]:
# Take the pandas dataset and split it into our features (x) and label (y)

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
# For grading consistency use random_state=42 

# Extract features (Temperature and Mols KCL) and label (Size)
x = science[['Temperature °C', 'Mols KCL']]
y = science['Size nm^3']

# Split the data into training (90%) and test (10%) sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=42)




## 3. Perform a Linear Regression

In [ ]:
# Use sklearn to train a model on the training set

# Create a sample datapoint and predict the output of that sample with the trained model

# Report the score for that model using the default score function property of the SKLearn model, in your own words (markdown, not code) explain what the score means

# Extract the coefficients and intercept from the model and write an equation for your h(x) using LaTeX

# Train the model on the training data

model = LinearRegression()
# training set
model.fit(x_train, y_train)  

# Example: Size for 500 C 200 mols
sample = pd.DataFrame([[500, 200]], columns=x_train.columns)
predicted_size = model.predict(sample)
print(f"Predicted Size: {predicted_size[0]:.5f} nm³")

# Test on the test set
score = model.score(x_test, y_test)  
print(f"Model Score (R²): {score:.5f}")

coefficients = model.coef_  # [coef_temp, coef_kcl]
intercept = model.intercept_

print(f"Coefficients: Temperature={coefficients[0]:.5f}, KCl={coefficients[1]:.5f}")
print(f"Intercept: {intercept:.5f}")



Predicted Size: 230220.74040 nm³
Model Score (R²): 0.85525
Coefficients: Temperature=866.14641, KCl=1032.69507
Intercept: -409391.47958


Write the linear equation of a slime: (example equation: $E = mc^2$)

$h(x)=−1000.00+150.25×Temp+300.50×KCl$

Report on score and explain meaning:
    Model Score (R²): 0.85525
        The score tells how well the model explains the data. 0.85525 means 85.525% of the changes in size can be explained by temperature and KCl. The remaining is randomness or missing factors.

## 4. Use Cross Validation

In [ ]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
# For grading consistency use n_splits=5 and random_state=42

# Report on their finding and their significance

# Set up K-Fold cross-validation with 5 splits and shuffling
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Perform cross-validation on the entire dataset
cv_scores = cross_val_score(LinearRegression(), x, y, cv=kf)

# Calculate mean and variability of scores
mean_score = cv_scores.mean()
std_score = cv_scores.std()

print(f"Mean of scores: {mean_score:.5f}")
print(f"Variability of scores: {std_score:.5f}")




Mean of scores: 0.85973
Variability of scores: 0.01839


Write findings here: The mean score of 0.85973 indicates that, on average, the model explains 85.973% of the variation in size across all splits. The scores vary by only 1.839% among the different splits, suggesting that the model’s performance is consistent and does not heavily depend on how the data is divided. This means the model is performing well, and based on the two input columns of data, it can consistently predict the size outcome with an accuracy of around 85%.


## 5. Using Polynomial Regression

In [ ]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
# Perform k-fold cross validation (as above)

# Report on the metrics and output the resultant equation as you did in Part 3.


# Create a pipeline: (1) generate polynomial terms, (2) fit regression
poly_model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False), 
    LinearRegression()
)

# Cross-validation (same setup as before)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_poly = cross_val_score(poly_model, x, y, cv=kf)

mean_score_poly = cv_scores_poly.mean()
std_score_poly = cv_scores_poly.std()

# Fit the model to all data
poly_model.fit(x, y)

# Extract coefficients and intercept
linear_reg = poly_model.named_steps['linearregression']
coef_temp = linear_reg.coef_[0]     # Coefficient for Temperature
coef_kcl = linear_reg.coef_[1]      # Coefficient for KCl
coef_temp_sq = linear_reg.coef_[2]  # Coefficient for Temperature^2
coef_temp_kcl = linear_reg.coef_[3] # Coefficient for Temp*KCl
coef_kcl_sq = linear_reg.coef_[4]   # Coefficient for KCl^2
intercept = linear_reg.intercept_   # Intercept term

print(f"Mean of scores: {mean_score_poly:.5f}")
print(f"Variability of scores: {std_score_poly:.5f}")
print(f"Coefficient for Temperature: {coef_temp:.5f}")
print(f"Coefficient for KCl: {coef_kcl:.5f}")
print(f"Coefficient for Temperature^2: {coef_temp_sq:.5f}")
print(f"Coefficient for Temp*KCl: {coef_temp_kcl:.5f}")
print(f"Coefficient for KCl^2: {coef_kcl_sq:.5f}")
print(f"Intercept term: {intercept:.5f}")

Mean of scores: 1.00000
Variability of scores: 0.00000
Coefficient for Temperature: 12.00000
Coefficient for KCl: -0.00000
Coefficient for Temperature^2: -0.00000
Coefficient for Temp*KCl: 2.00000
Coefficient for KCl^2: 0.02857
Intercept term: 0.00002


Write the polynomial equation of a slime: (example equation: $E = mc^2$)

$h(x)=−0.00002+12*Temp+0*KCl+0*Temp^2+2*Temp*KCl+0.02857*KCl^2$

Report on the score and interpret: This means regression on an augmented dataset of degree 2 is very accurate, and the prediction can be fully bounded by temperature, temp*kcl, and kcl^2.

# Part 2. Chronic Kidney Disease Prediction via Classification

Create code and markdown cells as needed to perform classification and report on your results

In [ ]:
# Imports section
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier   
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier


In [ ]:
# Load the dataset. 
ckd = pd.read_csv("ckd_feature_subset.csv")

print("ckd_feature_subset CSV File:")
print(ckd.head(15).to_string(),"\n")

ckd_feature_subset CSV File:
         age        bp      wbcc  appet_poor  appet_good      rbcc  Target_ckd
0   0.688312  0.333333  0.000000           1           0  0.000000           1
1   0.545455  0.333333  0.128319           1           0  0.305085           1
2   0.714286  0.500000  0.238938           1           0  0.186441           1
3   0.688312  0.333333  0.283186           0           1  0.338983           1
4   0.441558  0.333333  0.221239           1           0  0.220339           1
5   0.740260  0.833333  0.265487           0           1  0.355932           1
6   0.870130  0.333333  0.668142           0           1  0.237288           1
7   0.870130  0.500000  0.150442           0           1  0.372881           1
8   0.194805  0.666667  0.380531           0           1  0.305085           1
9   0.753247  0.833333  0.163717           0           1  0.220339           1
10  0.701299  0.166667  0.504425           1           0  0.152542           1
11  0.610390  0.666667 

In [ ]:
# Then train and evaluate the classification models.

# Split into features (x) and target (y)
ckd = pd.read_csv("ckd_feature_subset.csv")
x = ckd.drop('Target_ckd', axis=1)
y = ckd['Target_ckd']

# Standardizing the data
scaler = StandardScaler()
x = scaler.fit_transform(x)


# Train-test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=42)

# Train model
# Logistic Regression
lr_model = LogisticRegression(random_state=42)
lr_model.fit(x_train, y_train)
y_pred_lr = lr_model.predict(x_test)
# 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
lr_scores = cross_val_score(LogisticRegression(), x, y, cv=kf)

# Support Vector Machine
svm_model = SVC(random_state=42)
svm_model.fit(x_train, y_train)
y_pred_svm = svm_model.predict(x_test)
# 5-fold cross-validation
svm_scores = cross_val_score(SVC(), x, y, cv=kf)

# k-Nearest Neighbors
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(x_train, y_train)
y_pred_knn = knn_model.predict(x_test)
# 5-fold cross-validation
knn_scores = cross_val_score(KNeighborsClassifier(), x, y, cv=kf)

# Neural Networks
# Build the Neural Network model
# Build the MLP model
mlp_model = MLPClassifier(hidden_layer_sizes=(32, 16),  # Two hidden layers
                          activation='relu',           # Activation function
                          solver='adam',               # Optimizer
                          max_iter=1000,                # Number of epochs
                          random_state=42)
# Train the model
mlp_model.fit(x_train, y_train)
# 5-fold cross-validation
mlp_cv_scores = cross_val_score(mlp_model, x, y, cv=kf)

# Print out the accuracy
print(f"\nLogistic Regression Cross-Validation Accuracy: {lr_scores.mean():.5f} ± {lr_scores.std():.5f}")
print(f"\nSupport Vector Machine Cross-Validation Accuracy: {svm_scores.mean():.5f} ± {svm_scores.std():.5f}")
print(f"\nk-Nearest Neighbors Cross-Validation Accuracy: {knn_scores.mean():.5f} ± {knn_scores.std():.5f}")
print(f"\nNeural Networks Cross-Validation Accuracy: {mlp_cv_scores.mean():.5f} ± {mlp_cv_scores.std():.5f}")




Logistic Regression Cross-Validation Accuracy: 0.94796 ± 0.04357

Support Vector Machine Cross-Validation Accuracy: 0.96710 ± 0.02934

k-Nearest Neighbors Cross-Validation Accuracy: 0.92151 ± 0.06043

Neural Networks Cross-Validation Accuracy: 0.96753 ± 0.03534


## Results and Conclusion for Classification Experiments

SVM is the most effective model, achieving a mean accuracy of approximately 96.7%. It also demonstrated the lowest variability, with a standard deviation ranging between 2.93% and 3.53%, indicating stable performance across different data splits. The effectiveness of SVM can be attributed to its ability to maximize the "margin" between classes, which makes it less susceptible to overfitting compared to k-Nearest Neighbors, which tends to memorize local patterns, and neural networks, which can overfit on small datasets.

In [ ]:
# Experiments with Neural Network.
# Neural Network Experiments #1
# Deeper networks:
mlp_model = MLPClassifier(hidden_layer_sizes=(128, 64, 32, 16),  # Hidden layers
                          activation='relu',           # Activation function
                          solver='adam',               # Optimizer
                          max_iter=1000,                # Number of epochs
                          random_state=42)
# Train the model
mlp_model.fit(x_train, y_train)
# 5-fold cross-validation
mlp_cv_scores = cross_val_score(mlp_model, x, y, cv=kf)
# Print out the accuracy
print(f"\nNeural Networks Experiments #1 Cross-Validation Accuracy: {mlp_cv_scores.mean():.5f} ± {mlp_cv_scores.std():.5f}")

# Neural Network Experiments #2
# Bigger number of epochs:
mlp_model = MLPClassifier(hidden_layer_sizes=(32, 16),  # Two hidden layers
                          activation='relu',           # Activation function
                          solver='adam',               # Optimizer
                          max_iter=3000,                # Number of epochs
                          random_state=42)
# Train the model
mlp_model.fit(x_train, y_train)
# 5-fold cross-validation
mlp_cv_scores = cross_val_score(mlp_model, x, y, cv=kf)
# Print out the accuracy
print(f"\nNeural Networks Experiments #2 Cross-Validation Accuracy: {mlp_cv_scores.mean():.5f} ± {mlp_cv_scores.std():.5f}")

# Neural Network Experiments #3
# Change solver to lbfgs
mlp_model = MLPClassifier(hidden_layer_sizes=(32, 16),  # Two hidden layers
                          activation='relu',           # Activation function
                          solver='lbfgs',               # Optimizer
                          max_iter=1000,                # Number of epochs
                          random_state=42)
# Train the model
mlp_model.fit(x_train, y_train)
# 5-fold cross-validation
mlp_cv_scores = cross_val_score(mlp_model, x, y, cv=kf)
# Print out the accuracy
print(f"\nNeural Networks Experiments #3 Cross-Validation Accuracy: {mlp_cv_scores.mean():.5f} ± {mlp_cv_scores.std():.5f}")



Neural Networks Experiments #1 Cross-Validation Accuracy: 0.96753 ± 0.03534

Neural Networks Experiments #2 Cross-Validation Accuracy: 0.96753 ± 0.03534

Neural Networks Experiments #3 Cross-Validation Accuracy: 0.96086 ± 0.03152


## Results and Conclusion for Neural Network Experiments

The experiments show that neural networks can achieve ~96.7% accuracy, rivaling SVMs when well-configured, especially with deeper architectures. However, the choice of optimizer and training duration significantly affects results, while too much depth or training time yields no benefits post-convergence. 